# AutoML comparator runs (TPOT, mAML)

Launches the TPOT and mAML comparator arms against the same train/test splits,
metadata enrichment and search budget as the ritme and auto-sklearn arms, via
`submit_comparator(...)` from `src/launch_comparators.py`.

The data splits (`data_splits_*/train_val.pkl`, `test.pkl`) are produced by
`submit_model(...)` from `src/launch_models.py`; run those once first.

Run this notebook from the `ritme_usecases` env. Each worker runs in its own
environment, created once from the repo root:

```shell
mamba env create -f envs/tpot.yml -p $CONDA_ENVS/tpot_bench
mamba run -n tpot_bench pip install -e .

mamba env create -f envs/maml.yml -p $CONDA_ENVS/maml_bench
mamba run -n maml_bench pip install -e .
```

## Setup

In [ ]:
from src.launch_comparators import ensure_parquet_splits, submit_comparator

The comparator environments run NumPy 1.x and cannot read pickles written under
NumPy 2.x. Materialize parquet copies once before submitting anything.

In [ ]:
ensure_parquet_splits()

## Configuration

In [ ]:
# Where the comparator outputs and SLURM logs land.
LOGS_DIR = "comparators"

# Cluster account and node type come from .cluster.json (gitignored);
# see src/cluster_config.py. Leave as None to use those values.
SLURM_ACCOUNT = None

# Search budget (seconds) - matches the ritme and auto-sklearn arms.
TIME_BUDGET_S = 82800

# Per-job SLURM resources
CPUS = 50
MEM_PER_CPU_MB = 4096

common = dict(
    mode="slurm",
    logs_dir=LOGS_DIR,
    slurm_account=SLURM_ACCOUNT,
    total_time_s=TIME_BUDGET_S,
    cpus=CPUS,
    mem_per_cpu_mb=MEM_PER_CPU_MB,
)

## TPOT

One job per use case. The estimator family is pinned to match each use case's
ritme winner (`src/comparator_tpot.py:ESTIMATOR_FOR_USECASE`) while the
preprocessing operators stay searchable.

In [ ]:
# u1 needs a checkpoint dir and a larger per-evaluation cap: TPOT's stopit
# timeout raises asynchronously and can corrupt XGBoost's heap, ending the run.
# A crash is then recoverable with `--recover-from`.
for usecase in ("u1", "u2", "u3"):
    submit_comparator(
        usecase,
        method="tpot",
        max_eval_time_mins=120 if usecase == "u1" else 20,
        checkpoint_dir=(
            f"comparators/{usecase}_tpot_checkpoints" if usecase == "u1" else None
        ),
        **common,
    )

## mAML

Classification only, so use case 3 alone. This arm re-implements mAML's
published search space against the ritme split - see the module docstring of
`src/comparator_maml.py` for the deviations from the upstream CLI.

In [ ]:
submit_comparator("u3", method="maml", **common)